In [ ]:
!pip install transformers datasets torch scikit-learn --quiet

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dataset = load_dataset("imdb")
print(dataset)
print("\nSample:", dataset["train"][0])

In [ ]:
train_data = dataset["train"].shuffle(seed=42).select(range(500))
test_data  = dataset["test"].shuffle(seed=42).select(range(100))

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_tokenized = train_data.map(tokenize, batched=True)
test_tokenized  = test_data.map(tokenize, batched=True)

train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized  = test_tokenized.rename_column("label", "labels")

train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done.")

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
print("Model loaded.")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print("\nEvaluation Results:")
for key, value in results.items():
    print(f"  {key}: {round(value, 4)}")

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    label = "Positive" if pred == 1 else "Negative"
    print(f"Review: {text[:80]}...")
    print(f"Predicted Sentiment: {label}\n")

predict_sentiment("This movie was absolutely fantastic! The acting was superb.")
predict_sentiment("Terrible film, complete waste of time. Very disappointing.")
predict_sentiment("Average movie. Some good parts but overall nothing special.")

In [ ]:
model.save_pretrained("./bert-imdb-final")
tokenizer.save_pretrained("./bert-imdb-final")
print("Model and tokenizer saved to ./bert-imdb-final")

In [ ]:
from google.colab import files

notebook_path = "bert_imdb_finetuning.ipynb"

import json, os

nb_content = {
    "nbformat": 4,
    "nbformat_minor": 0,
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "name": "python3"},
        "language_info": {"name": "python"}
    },
    "cells": []
}

with open(notebook_path, "w") as f:
    json.dump(nb_content, f)

files.download(notebook_path)
print("Notebook downloaded. Upload this to your GitHub repo.")